# 2. Research Agent with Custom Tools

This notebook demonstrates building an iterative ReAct research agent equipped with web search and content summarization tools.

In [ ]:
import os
import asyncio
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

load_dotenv()

## 1. Search Tool with Tavily and DuckDuckGo Fallback

In [ ]:
class SearchResult(BaseModel):
    title: str
    url: str
    snippet: str

def duckduckgo_search(query: str, max_results: int = 4) -> List[SearchResult]:
    try:
        from duckduckgo_search import DDGS
        ddgs = DDGS()
        raw = list(ddgs.text(query, max_results=max_results))
        return [SearchResult(title=r.get('title', ''), url=r.get('href', ''), snippet=r.get('body', '')) for r in raw]
    except Exception as e:
        return []

async def search_web(query: str, max_results: int = 4) -> List[SearchResult]:
    tavily_key = os.getenv("TAVILY_API_KEY", "").strip()
    if tavily_key:
        try:
            from tavily import TavilyClient
            client = TavilyClient(api_key=tavily_key)
            resp = client.search(query=query, max_results=max_results)
            return [SearchResult(title=r.get('title', ''), url=r.get('url', ''), snippet=r.get('content', '')) for r in resp.get('results', [])]
        except Exception:
            pass
    return await asyncio.to_thread(duckduckgo_search, query, max_results)

## 2. Research Agent State and Nodes

In [ ]:
class ResearchNote(BaseModel):
    topic: str
    summary: str
    key_findings: List[str] = Field(default_factory=list)
    sources: List[Dict[str, str]] = Field(default_factory=list)

class WorkerState(BaseModel):
    topic: str
    search_queries: List[str] = Field(default_factory=list)
    depth: str = "in-depth"
    raw_results: List[Dict[str, Any]] = Field(default_factory=list)
    notes: List[ResearchNote] = Field(default_factory=list)
    iteration_count: int = 0
    status: str = "pending"

async def researcher_node(state: WorkerState) -> Dict[str, Any]:
    # Run web search
    if state.iteration_count == 0:
        queries = state.search_queries or [state.topic]
        all_results = []
        for q in queries[:2]:
            results = await search_web(q, max_results=3)
            for r in results:
                all_results.append({"title": r.title, "url": r.url, "snippet": r.snippet})
        return {"raw_results": all_results, "iteration_count": state.iteration_count + 1, "status": "searching"}
    
    # Synthesize findings
    llm = ChatOpenAI(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        openai_api_key=os.getenv("OPENAI_API_KEY"),
        openai_api_base=os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1"),
        temperature=0.2,
    )
    
    evidence = "\n".join([f"- [{r['title']}] ({r['url']}): {r['snippet']}" for r in state.raw_results])
    prompt = f"Synthesize research on: {state.topic}\n\nEvidence:\n{evidence}\n\nProduce a summary and 4 key findings."
    resp = await llm.ainvoke([SystemMessage(content="You are an expert research analyst."), HumanMessage(content=prompt)])
    
    note = ResearchNote(
        topic=state.topic,
        summary=resp.content,
        key_findings=[line.strip("- ") for line in resp.content.split("\n") if line.strip().startswith("-")],
        sources=[{"title": r["title"], "url": r["url"]} for r in state.raw_results if r.get("url")],
    )
    return {"notes": [note], "status": "completed", "iteration_count": state.iteration_count + 1}

def route_worker(state: WorkerState) -> str:
    if state.status == "completed" or state.iteration_count >= 2:
        return END
    return "researcher"

## 3. Build and Run Research Agent

In [ ]:
builder = StateGraph(WorkerState)
builder.add_node("researcher", researcher_node)
builder.add_edge(START, "researcher")
builder.add_conditional_edges("researcher", route_worker, {"researcher": "researcher", END: END})
research_graph = builder.compile()

sample_state = WorkerState(
    topic="Superconducting Qubits vs Trapped Ions in Quantum Computing",
    search_queries=["superconducting qubits vs trapped ions benchmarks 2026", "quantum computing modalities comparison"],
)
result = await research_graph.ainvoke(sample_state)
for note in result["notes"]:
    print(f"=== Research Note: {note.topic} ===")
    print(note.summary)
    print("Sources:", len(note.sources))